# LeetCode 188 — Best Time to Buy and Sell Stock IV (Hard) bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxProfit(self, k: int, prices: List[int]) -> int:

```

Đây là "trùm cuối" của chuỗi bài **mua bán cổ phiếu** trên LeetCode. Các bài trước trong chuỗi:

- **#121** — chỉ được làm **1** giao dịch (Easy).
- **#122** — được làm **không giới hạn** giao dịch (Medium).
- **#123** — được làm **tối đa 2** giao dịch (Hard).
- **#188** (bài này) — được làm **tối đa `k`** giao dịch, với `k` là tham số.

Bạn **không cần** học #123 trước: notebook này tự đủ, ôn lại #121 và #122 từ con số 0, rồi xây dần lên lời giải cho `k` bất kỳ.

---

## Sau notebook này, bạn sẽ nắm được

1. Hiểu đúng thế nào là "một giao dịch" và vì sao **tham lam** (greedy) kiểu "chọn `k` đoạn lời nhất" lại **sai**.
2. Ôn **#121** (giá nhỏ nhất chạy) và **#122** (cộng mọi đoạn tăng) — hai viên gạch nền.
3. **Cách 1 — Đệ quy thử mọi lựa chọn** (brute force): vẽ **cây quyết định** mua / bán / bỏ qua.
4. **Cách 2 — Memoization** (ghi nhớ) theo trạng thái `(ngày, số lượt còn lại, đang giữ hay không)`: từ hàm mũ xuống `O(n·k)`.
5. **Cách 3 — DP bảng** `dp[t][i]`: từ `O(k·n²)` cải tiến còn `O(k·n)` nhờ một **biến `best` chạy**.
6. **Cách 4 — Hai mảng `buy[t]` / `sell[t]`** chỉ `O(k)` bộ nhớ — **cách chính**, có bảng trace từng ngày.
7. **Cách 5 — Nhận xét `k ≥ n/2`**: khi đó bài biến thành #122, chỉ cần cộng mọi đoạn tăng.
8. (Nâng cao, tham khảo) ý tưởng **đỉnh–đáy + heap**.
9. Lỗi hay gặp, mẹo nhận dạng pattern, biến thể và 6 bài tập có đáp án.

## 1. Đề bài (theo LeetCode)

You are given an integer array `prices` where `prices[i]` is the price of a given stock on the `i`-th day, and an integer `k`.

Find the maximum profit you can achieve. You may complete **at most `k` transactions**: i.e. you may buy at most `k` times and sell at most `k` times.

**Note:** You may not engage in multiple transactions simultaneously (i.e., you must sell the stock before you buy again).

Dịch nhanh: cho mảng `prices` (giá cổ phiếu từng ngày) và số nguyên `k`. Tìm **lợi nhuận lớn nhất** nếu bạn được thực hiện **tối đa `k` giao dịch** (mua tối đa `k` lần, bán tối đa `k` lần). Không được ôm nhiều cổ phiếu cùng lúc: phải **bán xong** mới được **mua lại**.

**Example 1**

```text
Input:  k = 2, prices = [2,4,1]
Output: 2
Explanation: Buy on day 1 (price = 2) and sell on day 2 (price = 4), profit = 4-2 = 2.
```

**Example 2**

```text
Input:  k = 2, prices = [3,2,6,5,0,3]
Output: 7
Explanation: Buy on day 2 (price = 2) and sell on day 3 (price = 6), profit = 6-2 = 4.
Then buy on day 5 (price = 0) and sell on day 6 (price = 3), profit = 3-0 = 3.
```

**Constraints**

- `1 <= k <= 100`
- `1 <= prices.length <= 1000`
- `0 <= prices[i] <= 1000`

Ghi chú:

- Đề tiếng Anh đánh số ngày từ **1** ("day 1"), còn trong code Python ta đánh **index từ 0**. Vậy "day 2" của đề là `prices[1]`.
- Các phiên bản cũ của đề từng cho phép `k = 0` và mảng rỗng; code trong notebook này vẫn xử lý đúng các trường hợp đó (trả `0`) cho chắc.
- Kết quả không bao giờ âm: tệ nhất thì **không giao dịch gì** và lời `0`.

In [1]:
from typing import List   # LeetCode tự import sẵn; chạy ngoài LeetCode thì ta tự import

k1, prices1 = 2, [2, 4, 1]
k2, prices2 = 2, [3, 2, 6, 5, 0, 3]

for k, prices in [(k1, prices1), (k2, prices2)]:
    print(f"k = {k}, prices = {prices}, số ngày n = {len(prices)}")

k = 2, prices = [2, 4, 1], số ngày n = 3
k = 2, prices = [3, 2, 6, 5, 0, 3], số ngày n = 6


## 2. Hiểu đề bằng tay trước khi code

### "Một giao dịch" là gì?

Một **giao dịch** (transaction) = **một lần mua + một lần bán sau đó**. Lời của giao dịch = `giá bán − giá mua`.

Quy tắc:

1. Phải **mua trước, bán sau** (ngày bán ≥ ngày mua).
2. Tại mọi thời điểm chỉ ôm **tối đa 1 cổ phiếu**. Muốn mua lần 2 thì phải bán xong lần 1.
3. Được làm **tối đa** `k` giao dịch — làm ít hơn cũng được, kể cả `0` giao dịch.

Vẽ Example 2 ra cho dễ nhìn (mỗi `#` là 1 đơn vị giá):

In [2]:
def draw_prices(prices):
    """Vẽ biểu đồ cột nằm ngang: mỗi dòng là một ngày."""
    for i, p in enumerate(prices):
        print(f"ngày {i} | giá {p:>2} | " + "#" * p)

draw_prices([3, 2, 6, 5, 0, 3])

ngày 0 | giá  3 | ###
ngày 1 | giá  2 | ##
ngày 2 | giá  6 | ######
ngày 3 | giá  5 | #####
ngày 4 | giá  0 | 
ngày 5 | giá  3 | ###


### Liệt kê bằng tay Example 2 (`k = 2`)

Nhìn biểu đồ, có hai "dốc đi lên" rõ ràng:

| Giao dịch | Mua (index, giá) | Bán (index, giá) | Lời |
|---|---|---|---:|
| 1 | ngày 1, giá 2 | ngày 2, giá 6 | 4 |
| 2 | ngày 4, giá 0 | ngày 5, giá 3 | 3 |
| **Tổng** | | | **7** |

Thử phương án khác để thấy 7 là tốt nhất:

- Chỉ 1 giao dịch "mua 2 bán 6" = 4 (kém hơn).
- "Mua 2 (ngày 1), bán 3 (ngày 5)" = 1 (tệ).
- "Mua 0, bán 3" + "mua 2, bán 6" = 7 — đúng như trên, chỉ là thứ tự liệt kê khác.

Với Example 1 `[2,4,1]`, dù `k = 2` nhưng chỉ có **một** dốc đi lên (2 → 4), nên đáp án là 2. Nghĩa là **"tối đa k" không có nghĩa là bắt buộc dùng hết k**.

Nhắc lại các ký hiệu sẽ dùng suốt notebook:

- `n = len(prices)` — số ngày.
- `k` — số giao dịch tối đa.
- `t` — "giao dịch thứ `t`" hoặc "đang được phép dùng tối đa `t` giao dịch" (sẽ nói rõ ở từng cách).

## 3. Ôn nền tảng 1 — LeetCode #121: chỉ **1** giao dịch (`k = 1`)

### Cách người mới nghĩ ra đầu tiên: thử mọi cặp (ngày mua `i`, ngày bán `j > i`)

Với `prices = [7,1,5,3,6,4]`:

- mua 7 bán 1 → −6, mua 7 bán 5 → −2, …
- mua 1 bán 6 → **5** (lớn nhất).

Hai vòng `for` lồng nhau ⇒ `O(n²)`.

### Cải tiến: "giá nhỏ nhất từ đầu tới giờ" (running min)

Nếu **hôm nay bán**, thì tốt nhất là đã mua ở **ngày rẻ nhất trước đó**. Vậy chỉ cần nhớ một biến `min_price` = giá thấp nhất đã thấy, và mỗi ngày thử "bán hôm nay": `price - min_price`. Một vòng lặp ⇒ `O(n)`.

Hãy nhớ kỹ mẹo **"biến chạy giữ giá trị tốt nhất của quá khứ"** này — nó sẽ quay lại ở Cách 3 dưới tên `best`.

In [3]:
def max_profit_one_bruteforce(prices):
    """#121 bằng brute force: thử mọi cặp (mua i, bán j), O(n^2)."""
    best = 0
    for i in range(len(prices)):
        for j in range(i + 1, len(prices)):
            best = max(best, prices[j] - prices[i])
    return best


def max_profit_one(prices):
    """#121 bằng running min: O(n) thời gian, O(1) bộ nhớ."""
    min_price = float("inf")   # giá rẻ nhất đã thấy (ban đầu: chưa thấy gì => vô cực)
    best = 0
    for price in prices:
        min_price = min(min_price, price)        # có thể mua hôm nay
        best = max(best, price - min_price)      # thử bán hôm nay
    return best

for p in [[7, 1, 5, 3, 6, 4], [7, 6, 4, 3, 1], [2, 4, 1]]:
    print(p, "-> brute:", max_profit_one_bruteforce(p), "| running min:", max_profit_one(p))

[7, 1, 5, 3, 6, 4] -> brute: 5 | running min: 5
[7, 6, 4, 3, 1] -> brute: 0 | running min: 0
[2, 4, 1] -> brute: 2 | running min: 2


### Trace running min trên `[7,1,5,3,6,4]`

Cột `min_price` là giá rẻ nhất **tính cả hôm nay**; cột "bán hôm nay" là `price - min_price`.

In [4]:
def trace_one_transaction(prices):
    min_price = float("inf")
    best = 0
    print("ngày | giá | min_price | bán hôm nay | best")
    for i, price in enumerate(prices):
        min_price = min(min_price, price)
        today = price - min_price
        best = max(best, today)
        print(f"{i:>4} | {price:>3} | {min_price:>9} | {today:>11} | {best:>4}")
    return best

trace_one_transaction([7, 1, 5, 3, 6, 4])

ngày | giá | min_price | bán hôm nay | best
   0 |   7 |         7 |           0 |    0
   1 |   1 |         1 |           0 |    0
   2 |   5 |         1 |           4 |    4
   3 |   3 |         1 |           2 |    4
   4 |   6 |         1 |           5 |    5
   5 |   4 |         1 |           3 |    5


5

## 4. Ôn nền tảng 2 — LeetCode #122: **không giới hạn** giao dịch (`k = ∞`)

Khi được giao dịch bao nhiêu lần cũng được, ta cứ **ăn trọn mọi bước tăng giá**: hễ `prices[i] > prices[i-1]` thì cộng `prices[i] - prices[i-1]`.

### Vì sao cộng từng bước tăng nhỏ lại bằng một giao dịch dài?

Ví dụ giá `1 → 2 → 3 → 5`. Một giao dịch "mua 1, bán 5" lời `4`. Cộng từng bước: `(2−1) + (3−2) + (5−3) = 4`. Các số ở giữa **triệt tiêu nhau** (gọi là "tổng lồng kính" — telescoping sum). Nên một đoạn tăng liên tục dài bằng tổng các bước tăng nhỏ bên trong nó.

Còn các bước **giảm** thì ta né: bán trước khi giảm, mua lại ở đáy.

In [5]:
def max_profit_unlimited(prices):
    """#122: cộng mọi bước tăng giá. O(n) thời gian, O(1) bộ nhớ."""
    total = 0
    for i in range(1, len(prices)):
        if prices[i] > prices[i - 1]:
            total += prices[i] - prices[i - 1]
    return total

print(max_profit_unlimited([1, 2, 3, 5]))          # 4  (= 5 - 1)
print(max_profit_unlimited([7, 1, 5, 3, 6, 4]))    # 4 + 3 = 7
print(max_profit_unlimited([3, 2, 6, 5, 0, 3]))    # 4 + 3 = 7

4
7
7


### Đoạn tăng (run) — khái niệm sẽ dùng lại ở Cách 5

Một **đoạn tăng tối đại** (maximal increasing run) là một khúc giá đi lên liên tục mà không thể kéo dài thêm. Mỗi đoạn tăng ứng với **đúng 1 giao dịch** trong lời giải #122 (mua ở đáy đoạn, bán ở đỉnh đoạn).

Hàm dưới đây liệt kê các đoạn tăng dưới dạng `(ngày mua, ngày bán, lời)`.

In [6]:
def increasing_runs(prices):
    """Trả về list (ngày_mua, ngày_bán, lời) cho từng đoạn tăng tối đại."""
    runs = []
    n = len(prices)
    i = 0
    while i < n - 1:
        while i < n - 1 and prices[i + 1] <= prices[i]:   # trượt xuống tới đáy
            i += 1
        low = i
        while i < n - 1 and prices[i + 1] > prices[i]:    # leo lên tới đỉnh
            i += 1
        if i > low:
            runs.append((low, i, prices[i] - prices[low]))
    return runs

print(increasing_runs([3, 2, 6, 5, 0, 3]))
print(increasing_runs([1, 5, 4, 8]))
print(increasing_runs([7, 6, 4, 3, 1]))

[(1, 2, 4), (4, 5, 3)]
[(0, 1, 4), (2, 3, 4)]
[]


## 5. Khi `k` nằm **giữa** 1 và ∞ thì khó ở đâu?

Ý tưởng tham lam (greedy) tự nhiên: *"Liệt kê các đoạn tăng, rồi chọn `k` đoạn lời nhất."*

Thử với `prices = [1, 5, 4, 8]`, `k = 1`:

- Các đoạn tăng: `1 → 5` (lời 4) và `4 → 8` (lời 4).
- Greedy chọn 1 đoạn lời nhất ⇒ **4**.
- Nhưng đáp án đúng: **mua 1, bán 8** ⇒ **7**! Một giao dịch dài "nuốt" cả hai đoạn tăng, chỉ chịu thiệt cú giảm nhỏ `5 → 4`.

Tức là khi bị giới hạn số giao dịch, đôi khi nên **gộp** hai đoạn tăng (chấp nhận ôm qua một cú giảm) thay vì chọn riêng. Ngược lại, có lúc nên **tách** một đoạn dài. Không có quy tắc "nhìn một phát là biết" ⇒ ta cần một phương pháp **thử có hệ thống**: đệ quy → memo → DP.

In [7]:
def greedy_top_k_runs(k, prices):
    """SAI: chọn k đoạn tăng có lời lớn nhất."""
    profits = sorted((p for _, _, p in increasing_runs(prices)), reverse=True)
    return sum(profits[:k])

prices = [1, 5, 4, 8]
print("Greedy top-k đoạn tăng, k=1:", greedy_top_k_runs(1, prices))   # 4  (sai)
print("Đúng (#121, 1 giao dịch)  :", max_profit_one(prices))          # 7

Greedy top-k đoạn tăng, k=1: 4
Đúng (#121, 1 giao dịch)  : 7


## 6. Cách 1 — Đệ quy thử mọi lựa chọn (brute force)

### Kiến thức nền: đệ quy (recursion)

**Đệ quy** = một hàm **tự gọi lại chính nó** trên một bài toán **nhỏ hơn**. Mọi hàm đệ quy cần 2 phần:

1. **Base case** (điểm dừng): trường hợp đủ nhỏ để trả lời ngay, không gọi tiếp.
2. **Bước đệ quy**: đưa bài toán về bài nhỏ hơn và gọi lại.

Ví dụ mini: tính tổng một list bằng đệ quy — "tổng của `[3,1,4]` = `3` + tổng của `[1,4]`".

In [8]:
def sum_from(nums, i):
    """Tổng các phần tử nums[i], nums[i+1], ..., nums[-1]."""
    if i == len(nums):           # base case: không còn phần tử nào
        return 0
    return nums[i] + sum_from(nums, i + 1)   # bước đệ quy: bài nhỏ hơn bắt đầu từ i+1

print(sum_from([3, 1, 4], 0))    # 3 + (1 + (4 + 0)) = 8

8


### Mô hình hóa bài toán bằng **trạng thái** và **lựa chọn**

Ta đi qua từng ngày, từ ngày `0` tới ngày `n−1`. Tại mỗi ngày, mọi thứ ta cần biết để ra quyết định là:

- `i` — đang ở ngày nào;
- `t` — còn được **bán** bao nhiêu lần nữa (ta "trừ lượt" mỗi khi bán xong một giao dịch);
- `holding` — đang ôm cổ phiếu (`True`) hay đang cầm tiền (`False`).

Bộ ba `(i, t, holding)` gọi là **trạng thái** (state). Tại mỗi trạng thái có tối đa 2 lựa chọn:

| Đang ở trạng thái | Lựa chọn A | Lựa chọn B |
|---|---|---|
| `holding = False` (cầm tiền) | **bỏ qua** ngày này | **mua**: tiền `−prices[i]`, chuyển sang `holding = True` |
| `holding = True` (ôm cổ phiếu) | **bỏ qua** (tiếp tục giữ) | **bán**: tiền `+prices[i]`, `t` giảm 1, chuyển sang `holding = False` |

Gọi `f(i, t, holding)` = **số tiền lời thêm lớn nhất** có thể kiếm từ ngày `i` trở đi. Khi đó:

```text
f(i, t, holding) = max( bỏ qua: f(i+1, t, holding),
                         mua  : -prices[i] + f(i+1, t, True)      nếu chưa giữ
                         bán  : +prices[i] + f(i+1, t-1, False)   nếu đang giữ )
```

**Base case:**

- `i == n` (hết ngày) ⇒ `0`. Nếu lúc đó vẫn ôm cổ phiếu thì tiền mua đã bị trừ rồi, nên phương án đó tự thua, không cần xử lý riêng.
- `t == 0` (hết lượt bán) ⇒ `0`. Lúc này chắc chắn ta không ôm cổ phiếu, vì chỉ mua khi còn lượt.

Đáp án = `f(0, k, False)`.

In [9]:
def max_profit_recursive(k, prices):
    """Cách 1: thử mọi chuỗi lựa chọn mua / bán / bỏ qua. Thời gian hàm mũ."""
    n = len(prices)

    def f(i, t, holding):
        if i == n or t == 0:
            return 0
        skip = f(i + 1, t, holding)                       # bỏ qua ngày i
        if holding:
            act = prices[i] + f(i + 1, t - 1, False)      # bán hôm nay
        else:
            act = -prices[i] + f(i + 1, t, True)          # mua hôm nay
        return max(skip, act)

    return f(0, k, False)

print(max_profit_recursive(2, [2, 4, 1]))            # 2
print(max_profit_recursive(2, [3, 2, 6, 5, 0, 3]))   # 7
print(max_profit_recursive(1, [1, 5, 4, 8]))         # 7 (greedy sai ra 4)

2
7
7


### Trace: vẽ **cây quyết định** cho `k = 1`, `prices = [2, 4, 1]`

Mỗi dòng là một lời gọi `f(i, t, holding)`; thụt lề càng sâu là càng về sau trong thời gian. Dòng `=>` cho biết giá trị hàm trả về. Hãy dò theo nhánh "mua ngày 0 → bán ngày 1": `−2 + 4 + 0 = 2`.

In [10]:
def trace_decision_tree(k, prices):
    n = len(prices)

    def f(i, t, holding, depth, label):
        pad = "    " * depth
        state = "GIỮ" if holding else "tiền"
        if i == n or t == 0:
            print(f"{pad}{label} f({i},{t},{state}) = 0  [base case]")
            return 0
        print(f"{pad}{label} f({i},{t},{state})  giá hôm nay = {prices[i]}")
        skip = f(i + 1, t, holding, depth + 1, "bỏ qua:")
        if holding:
            act = prices[i] + f(i + 1, t - 1, False, depth + 1, f"bán +{prices[i]}:")
        else:
            act = -prices[i] + f(i + 1, t, True, depth + 1, f"mua -{prices[i]}:")
        res = max(skip, act)
        print(f"{pad}=> f({i},{t},{state}) = max({skip}, {act}) = {res}")
        return res

    return f(0, k, False, 0, "gốc:")

trace_decision_tree(1, [2, 4, 1])

gốc: f(0,1,tiền)  giá hôm nay = 2
    bỏ qua: f(1,1,tiền)  giá hôm nay = 4
        bỏ qua: f(2,1,tiền)  giá hôm nay = 1
            bỏ qua: f(3,1,tiền) = 0  [base case]
            mua -1: f(3,1,GIỮ) = 0  [base case]
        => f(2,1,tiền) = max(0, -1) = 0
        mua -4: f(2,1,GIỮ)  giá hôm nay = 1
            bỏ qua: f(3,1,GIỮ) = 0  [base case]
            bán +1: f(3,0,tiền) = 0  [base case]
        => f(2,1,GIỮ) = max(0, 1) = 1
    => f(1,1,tiền) = max(0, -3) = 0
    mua -2: f(1,1,GIỮ)  giá hôm nay = 4
        bỏ qua: f(2,1,GIỮ)  giá hôm nay = 1
            bỏ qua: f(3,1,GIỮ) = 0  [base case]
            bán +1: f(3,0,tiền) = 0  [base case]
        => f(2,1,GIỮ) = max(0, 1) = 1
        bán +4: f(2,0,tiền) = 0  [base case]
    => f(1,1,GIỮ) = max(1, 4) = 4
=> f(0,1,tiền) = max(0, 2) = 2


2

### Cách 1 vướng ở đâu?

Mỗi ngày (khi chưa chạm base case) đều **rẽ 2 nhánh** ⇒ số lời gọi cỡ `2^n`. Với `n = 1000` thì `2^1000` — vũ trụ cũng không đủ thời gian.

Hãy đếm số lời gọi thật sự để thấy nó "nổ" nhanh thế nào:

In [11]:
import random

def count_calls_recursive(k, prices):
    n = len(prices)
    calls = 0

    def f(i, t, holding):
        nonlocal calls          # nonlocal: cho phép sửa biến calls của hàm bên ngoài
        calls += 1
        if i == n or t == 0:
            return 0
        skip = f(i + 1, t, holding)
        act = prices[i] + f(i + 1, t - 1, False) if holding else -prices[i] + f(i + 1, t, True)
        return max(skip, act)

    f(0, k, False)
    return calls

random.seed(0)
for n in [4, 8, 12, 16]:
    prices = [random.randint(0, 9) for _ in range(n)]
    print(f"n = {n:>2}, k = 3 -> số lời gọi = {count_calls_recursive(3, prices):>7}")

n =  4, k = 3 -> số lời gọi =      31
n =  8, k = 3 -> số lời gọi =     493
n = 12, k = 3 -> số lời gọi =    5019
n = 16, k = 3 -> số lời gọi =   29785


### Quan sát then chốt: rất nhiều lời gọi **trùng trạng thái**

Nhìn lại cây quyết định ở trên: lời gọi `f(2,1,GIỮ)` (ngày 2, còn 1 lượt, đang ôm cổ phiếu) xuất hiện **2 lần** — một lần theo đường "bỏ qua ngày 0 → mua ngày 1", một lần theo đường "mua ngày 0 → giữ qua ngày 1". Cả hai lần đều tính ra cùng một kết quả `1`, vì giá trị của `f` **chỉ phụ thuộc vào trạng thái** `(i, t, holding)`, không phụ thuộc ta đã đi đường nào để tới đó. (Số tiền đã bỏ ra lúc mua được cộng ở tầng phía trên, không nằm trong `f`.)

Với input dài hơn, những lời gọi trùng như vậy nhân lên theo cấp số nhân.

Có bao nhiêu trạng thái **khác nhau**? `i` có `n + 1` giá trị, `t` có `k + 1` giá trị, `holding` có 2 giá trị ⇒ chỉ khoảng `2·(n+1)·(k+1)` trạng thái. Với `n = 1000`, `k = 100` là khoảng **200 nghìn** — nhỏ xíu so với `2^1000`.

⇒ Nếu **mỗi trạng thái chỉ tính một lần rồi ghi nhớ lại**, bài toán trở nên rất nhanh. Đó là Cách 2.

## 7. Cách 2 — Memoization theo `(i, t, holding)`

### Kiến thức nền: memoization (ghi nhớ kết quả)

**Memoization** = trước khi tính, xem "đã tính trạng thái này chưa?". Nếu rồi thì lấy kết quả cũ ra dùng; nếu chưa thì tính, rồi **lưu vào một dict** (key = trạng thái, value = kết quả).

Ví dụ mini kinh điển: số Fibonacci `fib(n) = fib(n−1) + fib(n−2)`. Bản đệ quy thường tính lại `fib(3)`, `fib(2)`… rất nhiều lần.

In [12]:
calls_plain = 0
def fib_plain(n):
    global calls_plain
    calls_plain += 1
    return n if n < 2 else fib_plain(n - 1) + fib_plain(n - 2)

calls_memo = 0
memo = {}                      # dict: key = n, value = fib(n)
def fib_memo(n):
    global calls_memo
    calls_memo += 1
    if n in memo:              # đã tính rồi -> trả luôn
        return memo[n]
    res = n if n < 2 else fib_memo(n - 1) + fib_memo(n - 2)
    memo[n] = res              # ghi nhớ
    return res

print("fib(25) =", fib_plain(25), "| số lời gọi không memo:", calls_plain)
print("fib(25) =", fib_memo(25), "| số lời gọi có memo   :", calls_memo)

fib(25) = 75025 | số lời gọi không memo: 242785
fib(25) = 75025 | số lời gọi có memo   : 49


### Cách viết gọn: `functools.lru_cache`

Python có sẵn "máy ghi nhớ" tự động: đặt `@lru_cache(maxsize=None)` lên trên hàm, Python sẽ tự lưu kết quả theo **bộ tham số** của hàm. Tham số phải **hashable** (băm được) như `int`, `bool`, `tuple` — không được là `list`.

Hàm `.cache_info()` cho biết đã "trúng" bộ nhớ (`hits`) bao nhiêu lần và đã lưu bao nhiêu trạng thái (`currsize`).

In [13]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fib_cached(n):
    return n if n < 2 else fib_cached(n - 1) + fib_cached(n - 2)

print(fib_cached(80))
print(fib_cached.cache_info())

23416728348467685
CacheInfo(hits=78, misses=81, maxsize=None, currsize=81)


### Áp dụng vào bài: thêm đúng **một dòng** `@lru_cache` vào Cách 1

Logic giữ nguyên 100%. Chỉ khác: mỗi trạng thái `(i, t, holding)` được tính **đúng một lần**.

Lưu ý: hàm `f` được định nghĩa **bên trong** `max_profit_memo`, nên mỗi lần gọi `max_profit_memo` sẽ có một bộ nhớ đệm mới — không bị lẫn dữ liệu giữa các test khác nhau.

In [14]:
import sys

def max_profit_memo(k, prices):
    """Cách 2: đệ quy + ghi nhớ theo (i, t, holding). O(n*k) thời gian và bộ nhớ."""
    n = len(prices)
    sys.setrecursionlimit(max(sys.getrecursionlimit(), 2 * n + 100))   # xem mục Lỗi 5

    @lru_cache(maxsize=None)
    def f(i, t, holding):
        if i == n or t == 0:
            return 0
        skip = f(i + 1, t, holding)
        if holding:
            act = prices[i] + f(i + 1, t - 1, False)
        else:
            act = -prices[i] + f(i + 1, t, True)
        return max(skip, act)

    ans = f(0, k, False)
    max_profit_memo.last_info = f.cache_info()    # lưu lại để in thống kê bên ngoài
    return ans

print(max_profit_memo(2, [2, 4, 1]))             # 2
print(max_profit_memo(2, [3, 2, 6, 5, 0, 3]))    # 7

2
7


### So sánh số lời gọi: brute force vs memo

`currsize` của cache = số trạng thái **khác nhau** đã tính. Nó không vượt quá `2·(n+1)·(k+1)`.

In [15]:
random.seed(0)
for n in [4, 8, 12, 16]:
    prices = [random.randint(0, 9) for _ in range(n)]
    a = max_profit_recursive(3, prices)
    b = max_profit_memo(3, prices)
    info = max_profit_memo.last_info
    print(f"n={n:>2}: kết quả {a}={b} | brute gọi {count_calls_recursive(3, prices):>6} lần"
          f" | memo lưu {info.currsize:>3} trạng thái (trần 2(n+1)(k+1) = {2*(n+1)*4})")

n= 4: kết quả 4=4 | brute gọi     31 lần | memo lưu  15 trạng thái (trần 2(n+1)(k+1) = 40)
n= 8: kết quả 7=7 | brute gọi    493 lần | memo lưu  42 trạng thái (trần 2(n+1)(k+1) = 72)
n=12: kết quả 15=15 | brute gọi   5019 lần | memo lưu  70 trạng thái (trần 2(n+1)(k+1) = 104)
n=16: kết quả 20=20 | brute gọi  29785 lần | memo lưu  98 trạng thái (trần 2(n+1)(k+1) = 136)


### Độ phức tạp Cách 2

- Số trạng thái: `O(n·k)` (nhân 2 cho `holding`, hằng số bỏ đi). Mỗi trạng thái làm `O(1)` việc (tối đa 2 lời gọi con đã được nhớ).
- **Thời gian:** `O(n·k)`. **Bộ nhớ:** `O(n·k)` cho cache + `O(n)` cho **ngăn xếp đệ quy** (call stack).

Điểm yếu còn lại:

1. Độ sâu đệ quy cỡ `n` — với `n = 1000`, Python mặc định giới hạn 1000 tầng nên **dễ bị `RecursionError`** (xem Lỗi 5). Ta phải nâng giới hạn bằng `sys.setrecursionlimit`.
2. Gọi hàm và tra cache trong Python khá chậm, tốn bộ nhớ.

⇒ Viết lại theo kiểu **từ dưới lên** (bottom-up) bằng vòng lặp: không đệ quy, kiểm soát được bộ nhớ. Đó là Cách 3 và Cách 4.

## 8. Cách 3 — DP bảng `dp[t][i]`

### Kiến thức nền: bảng 2 chiều (list lồng list)

DP bảng (tabulation) = điền kết quả các bài con vào một **bảng**, từ bài nhỏ tới bài lớn. Trong Python, bảng 2 chiều là một **list chứa các list**: `dp[t][i]` đọc là "hàng `t`, cột `i`".

Cách tạo đúng: `[[0] * cols for _ in range(rows)]` — mỗi hàng là một list **riêng**.

Cách tạo **sai** (rất hay gặp): `[[0] * cols] * rows` — tất cả các hàng là **cùng một list**, sửa một ô thì cả cột đổi theo!

In [16]:
rows, cols = 3, 4

good = [[0] * cols for _ in range(rows)]
good[1][2] = 9
print("Đúng:", good)

bad = [[0] * cols] * rows
bad[1][2] = 9
print("Sai :", bad, "  <- cả 3 hàng đều bị sửa vì là cùng một list")
print("good[0] is good[1]?", good[0] is good[1], "| bad[0] is bad[1]?", bad[0] is bad[1])

Đúng: [[0, 0, 0, 0], [0, 0, 9, 0], [0, 0, 0, 0]]
Sai : [[0, 0, 9, 0], [0, 0, 9, 0], [0, 0, 9, 0]]   <- cả 3 hàng đều bị sửa vì là cùng một list
good[0] is good[1]? False | bad[0] is bad[1]? True


### Định nghĩa ô bảng

`dp[t][i]` = **lợi nhuận lớn nhất** nếu chỉ xét các ngày `0..i` và được làm **tối đa `t`** giao dịch (lúc kết thúc ngày `i` đang cầm tiền, không ôm cổ phiếu).

Hàng `t = 0` toàn số 0 (không được giao dịch). Cột `i = 0` toàn số 0 (một ngày thì không lời được).

### Công thức chuyển (recurrence)

Tại ngày `i`, với tối đa `t` giao dịch, có 2 khả năng:

1. **Ngày `i` không bán gì** ⇒ lời giống như hết ngày `i−1`: `dp[t][i-1]`.
2. **Ngày `i` bán** để kết thúc giao dịch thứ `t`, giao dịch đó **mua ở ngày `j`** (`j < i`). Trước ngày `j` (tính tới hết ngày `j`) ta đã dùng tối đa `t−1` giao dịch: `dp[t-1][j]`. Tổng: `prices[i] - prices[j] + dp[t-1][j]`.

```text
dp[t][i] = max( dp[t][i-1],
                max over j < i of ( prices[i] - prices[j] + dp[t-1][j] ) )
```

(Dùng `dp[t-1][j]` — tức cho phép bán giao dịch trước ở ngày `j` rồi mua lại ngay ngày `j` — không sai: bán rồi mua lại cùng giá thì chẳng khác gì cứ giữ, nên không tạo ra lời "ảo".)

Ví dụ số, Example 2 `prices = [3,2,6,5,0,3]`, tính `dp[1][2]` (1 giao dịch, xét ngày 0..2):

- không bán ngày 2: `dp[1][1] = 0`;
- bán ngày 2 giá 6, mua ngày `j=0` (giá 3): `6 − 3 + dp[0][0] = 3`; mua ngày `j=1` (giá 2): `6 − 2 + dp[0][1] = 4`.
- ⇒ `dp[1][2] = max(0, 3, 4) = 4`.

In [17]:
def max_profit_dp_kn2(k, prices):
    """Cách 3a: DP bảng thẳng theo công thức, có vòng j bên trong. O(k * n^2)."""
    n = len(prices)
    if n == 0:
        return 0
    dp = [[0] * n for _ in range(k + 1)]
    for t in range(1, k + 1):
        for i in range(1, n):
            best = dp[t][i - 1]                      # ngày i không bán
            for j in range(i):                       # ngày i bán, mua ở ngày j
                best = max(best, prices[i] - prices[j] + dp[t - 1][j])
            dp[t][i] = best
    return dp[k][n - 1]

print(max_profit_dp_kn2(2, [2, 4, 1]))             # 2
print(max_profit_dp_kn2(2, [3, 2, 6, 5, 0, 3]))    # 7

2
7


### Cải tiến: bỏ vòng `j` bằng biến `best` chạy

Ba vòng lồng nhau ⇒ `O(k·n²)`. Với `k = 100`, `n = 1000` là `10^8` phép tính — quá chậm trong Python.

Nhìn kỹ phần trong vòng `j`:

```text
prices[i] - prices[j] + dp[t-1][j]  =  prices[i]  +  (dp[t-1][j] - prices[j])
                                         ^ không phụ thuộc j    ^ chỉ phụ thuộc j
```

Vậy ta chỉ cần `best = max over j < i of (dp[t-1][j] - prices[j])`. Khi `i` tăng lên 1, tập các `j` hợp lệ chỉ **thêm đúng một phần tử mới** là `j = i`. Nên không cần duyệt lại: giữ một biến `best` và cập nhật `best = max(best, dp[t-1][i] - prices[i])` sau mỗi bước — y hệt mẹo `min_price` chạy ở #121!

Ý nghĩa đời thường của `best`: "**số tiền tốt nhất còn trong túi nếu ta đã mua cổ phiếu cho giao dịch thứ `t`**" (lời từ `t−1` giao dịch trước trừ đi giá mua).

In [18]:
def max_profit_dp_kn(k, prices):
    """Cách 3b: DP bảng + biến best chạy. O(k*n) thời gian, O(k*n) bộ nhớ."""
    n = len(prices)
    if n == 0:
        return 0
    dp = [[0] * n for _ in range(k + 1)]
    for t in range(1, k + 1):
        best = dp[t - 1][0] - prices[0]              # chỉ có j = 0
        for i in range(1, n):
            dp[t][i] = max(dp[t][i - 1], prices[i] + best)
            best = max(best, dp[t - 1][i] - prices[i])   # thêm j = i cho vòng sau
    return dp[k][n - 1]

print(max_profit_dp_kn(2, [2, 4, 1]))             # 2
print(max_profit_dp_kn(2, [3, 2, 6, 5, 0, 3]))    # 7

2
7


### Trace bảng `dp` cho Example 2 (`k = 2`, `prices = [3,2,6,5,0,3]`)

Bảng kết quả (hàng = `t`, cột = ngày `i`):

| `t` \ `i` | 0 (3) | 1 (2) | 2 (6) | 3 (5) | 4 (0) | 5 (3) |
|---|---:|---:|---:|---:|---:|---:|
| 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 1 | 0 | 0 | 4 | 4 | 4 | 4 |
| 2 | 0 | 0 | 4 | 4 | 4 | **7** |

Đọc ô cuối: `dp[2][5] = max(dp[2][4], prices[5] + best) = max(4, 3 + 4) = 7`. Con số `best = 4` ở đó nghĩa là: "đã lời 4 từ giao dịch 1 (`dp[1][4] = 4`) rồi mua lại ở giá 0 ⇒ trong túi còn `4 − 0 = 4`".

Hàm trace dưới đây in từng bước của hàng `t = 2` (hàng `t = 1` tương tự) và in cả bảng.

In [19]:
def trace_dp_table(k, prices, show_row):
    n = len(prices)
    dp = [[0] * n for _ in range(k + 1)]
    for t in range(1, k + 1):
        best = dp[t - 1][0] - prices[0]
        if t == show_row:
            print(f"--- hàng t = {t}: best khởi đầu = dp[{t-1}][0] - prices[0] = {best}")
        for i in range(1, n):
            sell_today = prices[i] + best
            dp[t][i] = max(dp[t][i - 1], sell_today)
            if t == show_row:
                print(f"i={i} giá={prices[i]}: dp[{t}][{i}] = max({dp[t][i-1]}, {prices[i]}+{best}) = {dp[t][i]}"
                      f" | best mới = max({best}, dp[{t-1}][{i}]-{prices[i]} = {dp[t-1][i]-prices[i]})"
                      f" = {max(best, dp[t-1][i]-prices[i])}")
            best = max(best, dp[t - 1][i] - prices[i])
    print("Bảng dp:")
    for t in range(k + 1):
        print(f"  t={t}:", dp[t])
    return dp[k][n - 1]

trace_dp_table(2, [3, 2, 6, 5, 0, 3], show_row=2)

--- hàng t = 2: best khởi đầu = dp[1][0] - prices[0] = -3
i=1 giá=2: dp[2][1] = max(0, 2+-3) = 0 | best mới = max(-3, dp[1][1]-2 = -2) = -2
i=2 giá=6: dp[2][2] = max(0, 6+-2) = 4 | best mới = max(-2, dp[1][2]-6 = -2) = -2
i=3 giá=5: dp[2][3] = max(4, 5+-2) = 4 | best mới = max(-2, dp[1][3]-5 = -1) = -1
i=4 giá=0: dp[2][4] = max(4, 0+-1) = 4 | best mới = max(-1, dp[1][4]-0 = 4) = 4
i=5 giá=3: dp[2][5] = max(4, 3+4) = 7 | best mới = max(4, dp[1][5]-3 = 1) = 4
Bảng dp:
  t=0: [0, 0, 0, 0, 0, 0]
  t=1: [0, 0, 4, 4, 4, 4]
  t=2: [0, 0, 4, 4, 4, 7]


7

### Độ phức tạp Cách 3

- **3a** (có vòng `j`): thời gian `O(k·n²)`, bộ nhớ `O(k·n)`.
- **3b** (biến `best`): thời gian `O(k·n)`, bộ nhớ `O(k·n)`.

Quan sát thêm: hàng `t` chỉ cần **hàng `t−1`**. Vậy thật ra không cần giữ cả bảng. Cách 4 đi xa hơn một bước: đảo vòng lặp — **duyệt theo ngày ở ngoài**, và với mỗi ngày cập nhật **tất cả `k` giao dịch** cùng lúc, chỉ tốn `O(k)` bộ nhớ.

## 9. Cách 4 — Hai mảng `buy[t]` / `sell[t]` (cách chính, `O(k)` bộ nhớ)

### Hình dung bằng "ví tiền"

Bắt đầu với ví `0` đồng (được phép âm — coi như vay để mua). Với mỗi `t = 1..k`, ta theo dõi hai con số "tốt nhất tính tới hôm nay":

- `buy[t]` = số tiền **lớn nhất** trong ví sau khi đã **mua lần thứ `t`** (và đang ôm cổ phiếu đó).
- `sell[t]` = số tiền **lớn nhất** trong ví sau khi đã **bán lần thứ `t`** (đang cầm tiền).

Quy ước `sell[0] = 0`: chưa giao dịch gì thì ví có 0 đồng.

### Mỗi ngày với giá `p`, cập nhật theo thứ tự `t = 1, 2, …, k`:

```text
buy[t]  = max(buy[t],  sell[t-1] - p)   # giữ nguyên lần mua cũ, HOẶC mua hôm nay sau khi đã bán xong lần t-1
sell[t] = max(sell[t], buy[t]  + p)     # giữ nguyên lần bán cũ, HOẶC bán hôm nay cổ phiếu của lần mua t
```

Đáp án = `sell[k]`.

So với Cách 3: `buy[t]` chính là biến `best` của hàng `t`, còn `sell[t]` chính là `dp[t][i]` của ngày hiện tại. Ta chỉ đổi thứ tự hai vòng lặp, nên mỗi hàng chỉ cần nhớ **giá trị mới nhất**.

So với Cách 2: `buy[t]` ứng với trạng thái `holding = True`, `sell[t]` ứng với `holding = False` — đây là cùng một "máy trạng thái" (state machine), chỉ là tính xuôi theo ngày.

### Kiến thức nền: âm vô cực `float("-inf")`

Trước ngày đầu tiên ta **chưa thể** đã mua, nên `buy[t]` phải khởi tạo là một giá trị "tệ hơn mọi thứ" để lần `max` đầu tiên chắc chắn bị thay. Python có `float("-inf")` (âm vô cực):

- `max(float("-inf"), x)` luôn ra `x`;
- `float("-inf") + 5` vẫn là `-inf`.

Nếu lỡ khởi tạo `buy[t] = 0` thì như thể được **tặng cổ phiếu miễn phí** — sai (xem Lỗi 1).

In [20]:
NEG = float("-inf")
print(max(NEG, -3))          # -3
print(NEG + 5)               # -inf
print(NEG < -10**18)         # True: nhỏ hơn mọi số thật
print([NEG] * 3)             # khởi tạo mảng buy cho k = 2 (index 0 bỏ không dùng)

-3
-inf
True
[-inf, -inf, -inf]


### Thứ tự cập nhật trong cùng một ngày có gây lỗi không?

Trong cùng ngày, `sell[t]` dùng `buy[t]` **vừa được cập nhật hôm nay**, và `buy[t+1]` dùng `sell[t]` **vừa cập nhật hôm nay**. Điều đó tương đương "mua và bán trong cùng một ngày" (lời 0) hoặc "bán rồi mua lại cùng ngày" (cũng như không làm gì). Hai thao tác này **không tạo ra tiền ảo**, nên kết quả vẫn đúng. Ta sẽ kiểm chứng bằng test đối chiếu với brute force ở mục 11.

Code:

In [21]:
def max_profit_buy_sell(k, prices):
    """Cách 4: máy trạng thái buy[t]/sell[t]. O(n*k) thời gian, O(k) bộ nhớ."""
    buy = [float("-inf")] * (k + 1)    # buy[0] không dùng
    sell = [0] * (k + 1)               # sell[0] = 0 mãi mãi
    for p in prices:
        for t in range(1, k + 1):
            buy[t] = max(buy[t], sell[t - 1] - p)
            sell[t] = max(sell[t], buy[t] + p)
    return sell[k]

print(max_profit_buy_sell(2, [2, 4, 1]))             # 2
print(max_profit_buy_sell(2, [3, 2, 6, 5, 0, 3]))    # 7
print(max_profit_buy_sell(1, [1, 5, 4, 8]))          # 7

2
7
7


### Trace từng ngày — Example 2 (`k = 2`, `prices = [3,2,6,5,0,3]`)

Giá trị **sau khi** cập nhật xong mỗi ngày:

| Ngày | Giá | `buy[1]` | `sell[1]` | `buy[2]` | `sell[2]` | Chuyện gì xảy ra |
|---:|---:|---:|---:|---:|---:|---|
| 0 | 3 | −3 | 0 | −3 | 0 | lần đầu được mua: ví = −3 |
| 1 | 2 | **−2** | 0 | **−2** | 0 | giá rẻ hơn ⇒ đổi sang mua ở giá 2 |
| 2 | 6 | −2 | **4** | −2 | **4** | bán ở 6: `−2 + 6 = 4` |
| 3 | 5 | −2 | 4 | **−1** | 4 | `buy[2] = sell[1] − 5 = −1` tốt hơn −2 |
| 4 | 0 | **0** | 4 | **4** | 4 | `buy[2] = sell[1] − 0 = 4`: lời 4 rồi mua lại giá 0 |
| 5 | 3 | 0 | 4 | 4 | **7** | `sell[2] = buy[2] + 3 = 7` |

Đáp án `sell[2] = 7`. Để ý ngày 4: `buy[1]` cũng thành `0` (mua giá 0 cho giao dịch đầu tiên) — nó không được dùng trong đáp án cuối nhưng là một khả năng hợp lệ được giữ lại.

In [22]:
def trace_buy_sell(k, prices):
    buy = [float("-inf")] * (k + 1)
    sell = [0] * (k + 1)
    header = "ngày | giá | " + " | ".join(f"buy[{t}] sell[{t}]" for t in range(1, k + 1))
    print(header)
    for day, p in enumerate(prices):
        for t in range(1, k + 1):
            buy[t] = max(buy[t], sell[t - 1] - p)
            sell[t] = max(sell[t], buy[t] + p)
        cols = " | ".join(f"{buy[t]:>6} {sell[t]:>7}" for t in range(1, k + 1))
        print(f"{day:>4} | {p:>3} | {cols}")
    print("Đáp án sell[k] =", sell[k])
    return sell[k]

trace_buy_sell(2, [3, 2, 6, 5, 0, 3])

ngày | giá | buy[1] sell[1] | buy[2] sell[2]
   0 |   3 |     -3       0 |     -3       0
   1 |   2 |     -2       0 |     -2       0
   2 |   6 |     -2       4 |     -2       4
   3 |   5 |     -2       4 |     -1       4
   4 |   0 |      0       4 |      4       4
   5 |   3 |      0       4 |      4       7
Đáp án sell[k] = 7


7

### Trace thêm một input "khó": `k = 1` vs `k = 2` trên `[1, 5, 4, 8]`

Đây là input làm greedy thất bại. Hãy xem `buy`/`sell` tự "biết" lúc nào nên gộp (k = 1: mua 1 bán 8) và lúc nào nên tách (k = 2: 1→5 và 4→8, tổng 8).

In [23]:
trace_buy_sell(1, [1, 5, 4, 8])
print()
trace_buy_sell(2, [1, 5, 4, 8])

ngày | giá | buy[1] sell[1]
   0 |   1 |     -1       0
   1 |   5 |     -1       4
   2 |   4 |     -1       4
   3 |   8 |     -1       7
Đáp án sell[k] = 7

ngày | giá | buy[1] sell[1] | buy[2] sell[2]
   0 |   1 |     -1       0 |     -1       0
   1 |   5 |     -1       4 |     -1       4
   2 |   4 |     -1       4 |      0       4
   3 |   8 |     -1       7 |      0       8
Đáp án sell[k] = 8


8

### Vì sao đúng? (invariant bằng lời)

Sau khi xử lý xong ngày `d` (với mọi `t` từ 1 tới `k`):

- `sell[t]` = lợi nhuận lớn nhất có thể có **tới hết ngày `d`** khi đã hoàn tất **tối đa `t`** giao dịch và đang cầm tiền.
- `buy[t]` = số tiền lớn nhất trong ví **tới hết ngày `d`** khi đã hoàn tất tối đa `t−1` giao dịch **và đang ôm cổ phiếu của giao dịch thứ `t`** (`−∞` nếu chưa thể).

**Vì sao giữ được invariant khi sang ngày mới với giá `p`?** Mọi phương án "đang ôm cổ phiếu lần `t`" tới hôm nay chỉ có hai nguồn: đã ôm từ trước (`buy[t]` cũ) hoặc mua hôm nay sau khi đã bán xong lần `t−1` (`sell[t-1] − p`). Lấy `max` của hai nguồn là không bỏ sót. Tương tự với `sell[t]`: đã bán từ trước, hoặc bán hôm nay cổ phiếu của lần mua `t` (`buy[t] + p`). Mỗi giá trị đều ứng với một chuỗi giao dịch **hợp lệ thật sự** (mua trước bán sau, không ôm 2 cổ phiếu), nên cũng không trả nhầm số lớn hơn thực tế.

**Vì sao `sell[k]` là đáp án "tối đa k"** chứ không phải "đúng k"? Vì `sell[t]` khởi tạo bằng 0 cho mọi `t`, nghĩa là "chưa dùng hết lượt" vẫn hợp lệ. Hệ quả: `sell[1] ≤ sell[2] ≤ … ≤ sell[k]`, không cần lấy `max(sell)`.

### Độ phức tạp Cách 4

- **Thời gian:** `O(n·k)` — vòng ngoài `n` ngày, vòng trong `k` giao dịch, mỗi bước `O(1)`. Với `n = 1000`, `k = 100` là `10^5` bước, rất nhanh.
- **Bộ nhớ:** `O(k)` — chỉ hai mảng độ dài `k + 1`.

## 10. Cách 5 — Nhận xét `k ≥ n/2`: bài biến thành #122

### Một giao dịch có lời cần ít nhất 2 ngày

Lời giải tối ưu của #122 (không giới hạn) dùng đúng **một giao dịch cho mỗi đoạn tăng tối đại**. Mỗi đoạn tăng chiếm **ít nhất 2 ngày** (đáy và đỉnh), và các đoạn **không chung ngày** với nhau ⇒ số đoạn tăng ≤ `n // 2`.

Do đó: **nếu `k ≥ n // 2`**, giới hạn `k` không bao giờ "cản" lời giải #122 ⇒ đáp án = `max_profit_unlimited(prices)`, tính trong `O(n)`.

Ví dụ: `prices = [1,2,1,2,1,2]` (n = 6) có đúng 3 đoạn tăng = `6 // 2`. Đây là trường hợp "nhiều đoạn nhất có thể".

### Vì sao đáng biết?

- Với constraints hiện tại (`k ≤ 100`, `n ≤ 1000`) thì Cách 4 đã đủ nhanh, nhưng phím tắt này vẫn giúp bỏ qua vòng `k` khi `k` lớn.
- Ở các phiên bản đề cũ (hoặc khi phỏng vấn), `k` có thể **rất lớn** (ví dụ `10^9`). Khi đó tạo mảng `buy` độ dài `k + 1` sẽ **tràn bộ nhớ** — phím tắt này là bắt buộc.
- Nó cũng nhắc ta một tư duy hay: trước khi DP, hỏi xem **ràng buộc có thực sự "chặt" không**.

In [24]:
random.seed(5)
worst = 0
for _ in range(2000):
    n = random.randint(1, 12)
    prices = [random.randint(0, 9) for _ in range(n)]
    runs = len(increasing_runs(prices))
    assert runs <= n // 2
    worst = max(worst, runs - n // 2)
print("Số đoạn tăng luôn <= n // 2 (chênh lệch lớn nhất =", worst, ")")
print("Ví dụ đạt trần:", increasing_runs([1, 2, 1, 2, 1, 2]), "| n//2 =", 6 // 2)

# k >= n // 2 thì DP và #122 cho cùng kết quả
for k, p in [(3, [1, 2, 1, 2, 1, 2]), (50, [3, 2, 6, 5, 0, 3]), (2, [1, 3, 2, 8])]:
    print(f"k={k}, prices={p}: unlimited = {max_profit_unlimited(p)}, buy/sell = {max_profit_buy_sell(k, p)}")

Số đoạn tăng luôn <= n // 2 (chênh lệch lớn nhất = 0 )
Ví dụ đạt trần: [(0, 1, 1), (2, 3, 1), (4, 5, 1)] | n//2 = 3
k=3, prices=[1, 2, 1, 2, 1, 2]: unlimited = 3, buy/sell = 3
k=50, prices=[3, 2, 6, 5, 0, 3]: unlimited = 7, buy/sell = 7
k=2, prices=[1, 3, 2, 8]: unlimited = 8, buy/sell = 8


### Ghép lại thành lời giải chính

```text
nếu k >= n // 2:   trả về tổng mọi bước tăng         (O(n))
ngược lại:         chạy buy[t] / sell[t]             (O(n·k), với k < n/2)
```

Nhờ phím tắt, vòng `k` trong Cách 4 luôn có `k < n/2`, nên độ phức tạp tổng là `O(n · min(k, n))` thời gian và `O(min(k, n))` bộ nhớ.

In [25]:
def max_profit_main(k, prices):
    """Lời giải chính = Cách 5 (phím tắt) + Cách 4 (buy/sell)."""
    n = len(prices)
    if k >= n // 2:
        return max_profit_unlimited(prices)
    return max_profit_buy_sell(k, prices)

print(max_profit_main(2, [2, 4, 1]))             # n=3, n//2=1, k=2 >= 1 -> phím tắt -> 2
print(max_profit_main(2, [3, 2, 6, 5, 0, 3]))    # n//2=3, k=2 < 3 -> buy/sell -> 7

2
7


## 11. Code nộp LeetCode

Chuyển `max_profit_main` vào đúng signature `class Solution` / `def maxProfit(self, k, prices)`. Viết gộp vào một hàm để nộp không phụ thuộc hàm phụ bên ngoài.

In [26]:
class Solution:
    def maxProfit(self, k: int, prices: List[int]) -> int:
        n = len(prices)

        # Cách 5: k đủ lớn -> không giới hạn giao dịch -> cộng mọi bước tăng
        if k >= n // 2:
            return sum(max(0, prices[i] - prices[i - 1]) for i in range(1, n))

        # Cách 4: máy trạng thái buy[t] / sell[t]
        buy = [float("-inf")] * (k + 1)
        sell = [0] * (k + 1)
        for p in prices:
            for t in range(1, k + 1):
                buy[t] = max(buy[t], sell[t - 1] - p)
                sell[t] = max(sell[t], buy[t] + p)
        return sell[k]

In [27]:
sol = Solution()

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<38} -> {actual}  PASS")

check("Example 1", sol.maxProfit(2, [2, 4, 1]), 2)
check("Example 2", sol.maxProfit(2, [3, 2, 6, 5, 0, 3]), 7)
check("1 ngày", sol.maxProfit(1, [5]), 0)
check("giảm dần", sol.maxProfit(3, [7, 6, 4, 3, 1]), 0)
check("giá không đổi", sol.maxProfit(2, [4, 4, 4, 4]), 0)
check("k=1 phải gộp 2 đoạn tăng", sol.maxProfit(1, [1, 5, 4, 8]), 7)
check("k=2 tách 2 đoạn", sol.maxProfit(2, [1, 5, 4, 8]), 8)
check("k=2 kinh điển (#123)", sol.maxProfit(2, [3, 3, 5, 0, 0, 3, 1, 4]), 6)
check("k=2 chọn 2 trong 3 đoạn", sol.maxProfit(2, [1, 2, 4, 2, 5, 7, 2, 4, 9, 0]), 13)
check("k=4 < n//2=5 -> vòng DP, lấy đủ 3 đoạn", sol.maxProfit(4, [1, 2, 4, 2, 5, 7, 2, 4, 9, 0]), 15)
check("k rất lớn -> phím tắt", sol.maxProfit(100, [1, 2, 1, 2, 1, 2]), 3)
check("k=0 (đề cũ)", sol.maxProfit(0, [1, 3]), 0)
check("mảng rỗng (đề cũ)", sol.maxProfit(2, []), 0)

Example 1                              -> 2  PASS
Example 2                              -> 7  PASS
1 ngày                                 -> 0  PASS
giảm dần                               -> 0  PASS
giá không đổi                          -> 0  PASS
k=1 phải gộp 2 đoạn tăng               -> 7  PASS
k=2 tách 2 đoạn                        -> 8  PASS
k=2 kinh điển (#123)                   -> 6  PASS
k=2 chọn 2 trong 3 đoạn                -> 13  PASS
k=4 < n//2=5 -> vòng DP, lấy đủ 3 đoạn -> 15  PASS
k rất lớn -> phím tắt                  -> 3  PASS
k=0 (đề cũ)                            -> 0  PASS
mảng rỗng (đề cũ)                      -> 0  PASS


### Kiểm thử đối chiếu chéo tất cả các cách

Sinh ngẫu nhiên nhiều input nhỏ (cố định `random.seed` để lần chạy nào cũng giống nhau) và so **tất cả** các cách với brute force đệ quy (Cách 1). Brute force chậm nhưng **đơn giản nên dễ tin là đúng** — dùng nó làm "trọng tài".

In [28]:
random.seed(188)
methods = {
    "memo": max_profit_memo,
    "dp_kn2": max_profit_dp_kn2,
    "dp_kn": max_profit_dp_kn,
    "buy_sell": max_profit_buy_sell,
    "main": max_profit_main,
    "Solution": lambda k, p: Solution().maxProfit(k, p),
}
tests = 0
for _ in range(600):
    n = random.randint(1, 10)
    prices = [random.randint(0, 10) for _ in range(n)]
    k = random.randint(1, 6)
    expected = max_profit_recursive(k, prices)
    for name, fn in methods.items():
        got = fn(k, prices)
        assert got == expected, (name, k, prices, got, expected)
    tests += 1
print(f"{tests} bộ test ngẫu nhiên: cả {len(methods)} cách đều khớp brute force")

# Thêm: input đúng cỡ trần constraints (n = 1000) — chỉ chạy các cách O(n*k)
import time

random.seed(7)
big = [random.randint(0, 1000) for _ in range(1000)]   # n = 1000 (trần constraints)
for k in [1, 2, 10, 100]:
    start = time.perf_counter()
    a = Solution().maxProfit(k, big)
    elapsed = time.perf_counter() - start
    b = max_profit_dp_kn(k, big)
    assert a == b
    print(f"k={k:>3}: lời = {a:>6} | khớp dp_kn | thời gian < 1 giây: {elapsed < 1}")

600 bộ test ngẫu nhiên: cả 6 cách đều khớp brute force
k=  1: lời =    999 | khớp dp_kn | thời gian < 1 giây: True
k=  2: lời =   1995 | khớp dp_kn | thời gian < 1 giây: True
k= 10: lời =   9876 | khớp dp_kn | thời gian < 1 giây: True


k=100: lời =  84978 | khớp dp_kn | thời gian < 1 giây: True


## 12. So sánh các cách

Ký hiệu: `n` = số ngày, `k` = số giao dịch tối đa.

| Cách | Ý tưởng | Thời gian | Bộ nhớ | Ghi chú |
|---|---|---|---|---|
| 1. Đệ quy brute force | thử mua / bán / bỏ qua mỗi ngày | `O(2^n)` | `O(n)` (stack) | dễ nghĩ, dùng làm trọng tài để test |
| 2. Memo `(i, t, holding)` | ghi nhớ trạng thái trùng | `O(n·k)` | `O(n·k)` + stack `O(n)` | phải nâng recursion limit |
| 3a. DP bảng + vòng `j` | `dp[t][i]` theo ngày bán / ngày mua | `O(k·n²)` | `O(k·n)` | dễ hiểu công thức, chậm |
| 3b. DP bảng + `best` | tách `prices[i] + (dp[t-1][j] - prices[j])` | `O(k·n)` | `O(k·n)` | mẹo "max chạy" như #121 |
| **4. `buy[t]` / `sell[t]`** | máy trạng thái, duyệt theo ngày | `O(n·k)` | **`O(k)`** | **cách chính**, gọn nhất |
| 5. Phím tắt `k ≥ n/2` | bài trở thành #122 | `O(n)` | `O(1)` | luôn ghép với cách 4 |
| (Nâng cao) đỉnh–đáy + heap | tách lời thành các "gói" độc lập | `O(n log n)` | `O(n)` | xem mục 15 |

Thứ tự tư duy nên nhớ: **cây quyết định → nhận ra trạng thái trùng → memo → viết lại từ dưới lên → nén bộ nhớ → kiểm tra ràng buộc có chặt không**. Đây cũng là "lộ trình chuẩn" cho rất nhiều bài DP khác.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Khởi tạo `buy[t] = 0` thay vì `-inf`

`buy[t] = 0` nghĩa là "đã ôm cổ phiếu mà ví vẫn 0 đồng" — tức là được **tặng** cổ phiếu. Thuật toán sẽ "bán" món quà đó và báo lời ảo.

In [29]:
def wrong_buy_init_zero(k, prices):
    buy = [0] * (k + 1)                 # SAI: phải là float("-inf")
    sell = [0] * (k + 1)
    for p in prices:
        for t in range(1, k + 1):
            buy[t] = max(buy[t], sell[t - 1] - p)
            sell[t] = max(sell[t], buy[t] + p)
    return sell[k]

print("Sai :", wrong_buy_init_zero(2, [2, 4, 1]))     # 4: "bán" cổ phiếu được tặng giá 4
print("Đúng:", max_profit_buy_sell(2, [2, 4, 1]))     # 2

Sai : 4
Đúng: 2


### Lỗi 2: Lệch chỉ số — `sell[t]` dùng `buy[t-1]`

Bán ở giao dịch `t` phải bán **cổ phiếu của lần mua `t`**, tức `buy[t] + p`. Viết `buy[t-1] + p` làm mọi thứ "lệch một lượt": thực chất chỉ cho phép `k − 1` giao dịch (và `buy[0] = -inf` khiến `sell[1]` không bao giờ tăng).

In [30]:
def wrong_index_shift(k, prices):
    buy = [float("-inf")] * (k + 1)
    sell = [0] * (k + 1)
    for p in prices:
        for t in range(1, k + 1):
            buy[t] = max(buy[t], sell[t - 1] - p)
            sell[t] = max(sell[t], buy[t - 1] + p)    # SAI: phải là buy[t]
    return sell[k]

print("Sai :", wrong_index_shift(1, [2, 4, 1]), "| Đúng:", max_profit_buy_sell(1, [2, 4, 1]))
print("Sai :", wrong_index_shift(2, [3, 2, 6, 5, 0, 3]), "| Đúng:", max_profit_buy_sell(2, [3, 2, 6, 5, 0, 3]))

Sai : 0 | Đúng: 2
Sai : 4 | Đúng: 7


### Lỗi 3: Tạo bảng DP bằng `[[0] * n] * (k + 1)`

Như đã thấy ở mục 8, cách này làm mọi hàng là **cùng một list**. Trong DP, khi ghi `dp[t][i]` ta đồng thời ghi đè `dp[t-1][i]` — dữ liệu hàng trước bị phá. Có thể ra kết quả sai (như dưới đây), hoặc tệ hơn là **đúng tình cờ** trên vài test rồi sai khi nộp.

In [31]:
def wrong_aliased_table(k, prices):
    n = len(prices)
    dp = [[0] * n] * (k + 1)                  # SAI: các hàng dùng chung một list
    for t in range(1, k + 1):
        for i in range(1, n):
            best = dp[t][i - 1]
            for j in range(i):
                best = max(best, prices[i] - prices[j] + dp[t - 1][j])
            dp[t][i] = best
    return dp[k][n - 1]

for k, p in [(1, [1, 5, 4, 8]), (2, [3, 2, 6, 5, 0, 3]), (1, [1, 2, 1, 2, 1, 2])]:
    print(f"k={k}, {p}: bảng dùng chung = {wrong_aliased_table(k, p)}, đúng = {max_profit_dp_kn2(k, p)}")

k=1, [1, 5, 4, 8]: bảng dùng chung = 8, đúng = 7
k=2, [3, 2, 6, 5, 0, 3]: bảng dùng chung = 7, đúng = 7
k=1, [1, 2, 1, 2, 1, 2]: bảng dùng chung = 3, đúng = 1


### Lỗi 4: Dùng greedy "chọn `k` đoạn tăng lớn nhất"

Đã thấy ở mục 5: `[1,5,4,8]`, `k = 1` greedy ra 4 nhưng đáp án 7. Greedy không biết **gộp** hai đoạn tăng qua một cú giảm nhỏ. (Ý tưởng đỉnh–đáy + heap ở mục 15 chính là cách "vá" greedy này một cách đúng đắn — nhưng phức tạp hơn nhiều.)

### Lỗi 5: Memo đệ quy bị `RecursionError` khi `n = 1000`

Python mặc định chỉ cho khoảng 1000 tầng đệ quy. Cách 2 gọi `f(i)` → `f(i+1)` → … sâu tới `n` tầng ⇒ với `n = 1000` là vượt. Cách sửa: `sys.setrecursionlimit(...)` (như trong `max_profit_memo`), hoặc tốt hơn là viết bottom-up (Cách 3/4).

In [32]:
def memo_without_limit_fix(k, prices):
    n = len(prices)

    @lru_cache(maxsize=None)
    def f(i, t, holding):
        if i == n or t == 0:
            return 0
        skip = f(i + 1, t, holding)
        act = prices[i] + f(i + 1, t - 1, False) if holding else -prices[i] + f(i + 1, t, True)
        return max(skip, act)

    return f(0, k, False)

old_limit = sys.getrecursionlimit()
sys.setrecursionlimit(1000)                       # giới hạn mặc định của Python
long_prices = [i % 7 for i in range(1000)]
try:
    memo_without_limit_fix(5, long_prices)
    print("Không lỗi (môi trường này cho phép đệ quy sâu hơn)")
except RecursionError as e:
    print("RecursionError:", e)
sys.setrecursionlimit(max(old_limit, 3000))
print("Sau khi nâng giới hạn, max_profit_memo =", max_profit_memo(5, long_prices),
      "| buy/sell =", max_profit_buy_sell(5, long_prices))

RecursionError: maximum recursion depth exceeded
Sau khi nâng giới hạn, max_profit_memo = 30 | buy/sell = 30


### Lỗi 6 (nhẹ hơn nhưng hay gặp): quên trường hợp "không giao dịch"

- Trả về `max(sell)` hay `sell[k]` đều được vì `sell` tăng dần theo `t` — nhưng nếu khởi tạo `sell[t] = -inf` (nghĩ rằng "chưa bán thì không có giá trị") thì sẽ mất trường hợp "dùng ít hơn `k` giao dịch". Giữ `sell[t] = 0`.
- Với mảng giảm dần, đáp án phải là `0`, không phải số âm.

## 14. Mẹo nhận dạng pattern

Gặp đề có các dấu hiệu sau ⇒ nghĩ ngay tới **DP máy trạng thái** (state machine DP) giống bài này:

- Đi tuần tự theo thời gian (ngày, bước, vị trí), mỗi bước có **vài hành động** (mua / bán / nghỉ, bật / tắt, …).
- Có một **giới hạn đếm được** (tối đa `k` lần, cooldown 1 ngày, phí mỗi lần…).
- "Tham lam" nghe hợp lý nhưng có phản ví dụ nhỏ.

Công thức suy nghĩ 4 bước:

1. **Trạng thái** gồm những gì? (ở đây: ngày `i`, số lượt `t`, có đang giữ hay không).
2. **Hành động** và chuyển trạng thái ra sao? (mua: `−p`, bán: `+p` và `t−1`, bỏ qua).
3. Viết đệ quy + memo cho chắc đúng.
4. Đổi sang bottom-up, rồi **nén** chiều không cần nhớ (ở đây bỏ chiều ngày ⇒ `buy[t]`, `sell[t]`).

Thêm một mẹo con: thấy biểu thức dạng `max over j < i of (A[i] + B[j])` ⇒ tách ra, giữ `max B[j]` **chạy** ⇒ bớt một vòng lặp.

## 15. Biến thể và bài liên quan

| Bài | Khác gì | Điều chỉnh từ `buy`/`sell` |
|---|---|---|
| #121 Best Time I | `k = 1` | chỉ `buy1`, `sell1` (chính là running min) |
| #122 Best Time II | `k = ∞` | cộng mọi bước tăng, hoặc một cặp `buy`, `sell` không có chỉ số `t` |
| #123 Best Time III | `k = 2` | `buy1, sell1, buy2, sell2` — đúng Cách 4 với `k = 2` |
| #309 With Cooldown | không giới hạn, nhưng bán xong phải nghỉ 1 ngày | `buy` dùng `sell` của **hôm kia** |
| #714 With Transaction Fee | không giới hạn, mỗi lần bán trả phí `fee` | `sell = max(sell, buy + p - fee)` |

### (Nâng cao, tham khảo) Ý tưởng "đỉnh–đáy + heap" — `O(n log n)`

Còn một hướng nhìn rất đẹp, tách hẳn khỏi `k`:

1. Duyệt giá, tìm lần lượt từng cặp **(đáy `v`, đỉnh `p`)** của các đoạn tăng.
2. Giữ một **stack** các cặp chưa "chốt". Khi gặp cặp mới:
   - Nếu đáy mới **thấp hơn** đáy của cặp trên đỉnh stack ⇒ cặp cũ không thể gộp với gì phía sau tốt hơn, **chốt** lợi nhuận `đỉnh − đáy` của nó vào danh sách `profits`.
   - Nếu đỉnh mới **cao hơn hoặc bằng** đỉnh của cặp trên stack (và đáy cũ thấp hơn) ⇒ có hai lựa chọn: hai giao dịch riêng, hoặc một giao dịch gộp. Ta **tách** tổng lợi nhuận thành hai "gói": gói `đỉnh_cũ − đáy_mới` được chốt vào `profits`, còn cặp `(đáy_cũ, đỉnh_mới)` tiếp tục sống trên stack. Hay ở chỗ: lấy 1 gói = làm giao dịch gộp; lấy cả 2 gói = làm 2 giao dịch riêng.
3. Cuối cùng chốt hết stack. Đáp án = **tổng `k` gói lớn nhất** trong `profits` — lấy bằng **heap**.

Kiến thức nền mini: `heapq.nlargest(k, xs)` trả `k` phần tử lớn nhất của `xs` (dùng heap bên trong).

In [33]:
import heapq

print(heapq.nlargest(2, [4, 1, 7, 3]))     # [7, 4]
print(heapq.nlargest(5, [4, 1]))           # [4, 1]  (ít hơn k phần tử thì lấy hết)

[7, 4]
[4, 1]


In [34]:
def max_profit_peak_valley_heap(k, prices):
    """(Nâng cao) Tách lợi nhuận thành các gói độc lập, lấy k gói lớn nhất."""
    n = len(prices)
    stack, profits = [], []        # stack chứa cặp (index đáy, index đỉnh)
    v, p = 0, -1
    while True:
        v = p + 1
        while v + 1 < n and prices[v] >= prices[v + 1]:   # tìm đáy
            v += 1
        p = v
        while p + 1 < n and prices[p] <= prices[p + 1]:   # tìm đỉnh
            p += 1
        if v >= p:                                         # hết đoạn tăng
            break
        while stack and prices[v] < prices[stack[-1][0]]:  # đáy mới thấp hơn -> chốt cặp cũ
            sv, sp = stack.pop()
            profits.append(prices[sp] - prices[sv])
        while stack and prices[p] >= prices[stack[-1][1]]: # đỉnh mới cao hơn -> tách gói
            sv, sp = stack.pop()
            profits.append(prices[sp] - prices[v])
            v = sv
        stack.append((v, p))
    while stack:
        sv, sp = stack.pop()
        profits.append(prices[sp] - prices[sv])
    return sum(heapq.nlargest(k, profits))

print(max_profit_peak_valley_heap(1, [1, 5, 4, 8]))           # 7: gói 7 (1->8) và gói 1 (5-4)
print(max_profit_peak_valley_heap(2, [1, 5, 4, 8]))           # 8 = 7 + 1
print(max_profit_peak_valley_heap(2, [3, 2, 6, 5, 0, 3]))     # 7

random.seed(42)
for _ in range(3000):
    n = random.randint(1, 12)
    pr = [random.randint(0, 9) for _ in range(n)]
    kk = random.randint(1, 6)
    assert max_profit_peak_valley_heap(kk, pr) == max_profit_buy_sell(kk, pr)
print("Đỉnh–đáy + heap khớp buy/sell trên 3000 test ngẫu nhiên")

7
8
7


Đỉnh–đáy + heap khớp buy/sell trên 3000 test ngẫu nhiên


Với `[1,5,4,8]`: hai cặp (đáy 1, đỉnh 5) và (đáy 4, đỉnh 8). Đỉnh mới 8 ≥ đỉnh cũ 5 ⇒ tách thành gói `5 − 4 = 1` và cặp gộp (1, 8) = gói 7. `k = 1` lấy gói 7; `k = 2` lấy `7 + 1 = 8` — đúng bằng "1→5 và 4→8".

Bạn **không cần** thuộc cách này để qua bài; Cách 4 + Cách 5 là đủ và dễ viết đúng hơn nhiều trong phỏng vấn.

## 16. Bài tập vận dụng

Làm theo thứ tự. Các cell TODO chỉ **định nghĩa** hàm — test để ở dạng comment, tự bỏ comment khi làm xong. Đáp án ở **cuối notebook**.

### Bài 1 — Tính tay bảng `buy`/`sell`

Với `k = 2`, `prices = [1, 3, 2, 8]`, hãy tự điền bảng (giá trị **sau** mỗi ngày):

| Ngày | Giá | `buy[1]` | `sell[1]` | `buy[2]` | `sell[2]` |
|---:|---:|---:|---:|---:|---:|
| 0 | 1 | ? | ? | ? | ? |
| 1 | 3 | ? | ? | ? | ? |
| 2 | 2 | ? | ? | ? | ? |
| 3 | 8 | ? | ? | ? | ? |

Đáp án cuối là bao nhiêu? Nếu `k = 1` thì sao?

### Bài 2 — Đoán output (không chạy code)

Đoán `Solution().maxProfit(k, prices)` cho:

1. `k = 1`, `[3, 3, 3]`
2. `k = 5`, `[1, 2, 1, 2, 1, 2]`
3. `k = 1`, `[1, 5, 4, 8]`
4. `k = 2`, `[1, 5, 4, 8]`
5. `k = 1`, `[5, 1, 9, 0, 10]`

Với câu 2, chương trình đi qua nhánh phím tắt `k >= n // 2` hay qua vòng `buy/sell`?

### Bài 3 — Điền TODO: Cách 4 `buy`/`sell`

Hoàn thiện hàm theo đúng máy trạng thái đã học.

In [35]:
def exercise_buy_sell(k, prices):
    buy = None    # TODO 1: mảng k+1 phần tử, khởi tạo âm vô cực
    sell = None   # TODO 2: mảng k+1 phần tử, khởi tạo 0
    for p in prices:
        for t in range(1, k + 1):
            # TODO 3: cập nhật buy[t] từ sell[t-1] và giá p
            # TODO 4: cập nhật sell[t] từ buy[t] và giá p
            pass
    raise NotImplementedError  # TODO 5: trả về đáp án

# Tự làm xong rồi mới mở test:
# assert exercise_buy_sell(2, [2, 4, 1]) == 2
# assert exercise_buy_sell(2, [3, 2, 6, 5, 0, 3]) == 7
# assert exercise_buy_sell(1, [1, 5, 4, 8]) == 7

### Bài 4 — Tự viết lại Cách 2 (đệ quy + memo) theo cách đếm lượt khác

Ở Cách 2 ta trừ lượt `t` **khi bán**. Hãy viết lại sao cho trừ lượt **khi mua**: chỉ được mua nếu `t > 0`, mua xong `t` giảm 1, bán không đổi `t`. Cẩn thận base case: lúc `t == 0` vẫn có thể **đang giữ** cổ phiếu và cần bán!

In [36]:
def exercise_memo_count_on_buy(k, prices):
    # TODO: định nghĩa f(i, t, holding) với @lru_cache, trừ t khi MUA
    # TODO: base case chỉ là i == len(prices)
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# assert exercise_memo_count_on_buy(2, [2, 4, 1]) == 2
# assert exercise_memo_count_on_buy(2, [3, 2, 6, 5, 0, 3]) == 7
# assert exercise_memo_count_on_buy(1, [1, 5, 4, 8]) == 7

### Bài 5 — Biến thể: tối đa `k` giao dịch **và** mỗi lần bán mất phí `fee`

Viết `exercise_k_with_fee(k, prices, fee)`. Ví dụ `k = 2`, `prices = [1, 3, 2, 8, 4, 9]`, `fee = 2`:

- mua 1 bán 8: `7 − 2 = 5`; mua 4 bán 9: `5 − 2 = 3` ⇒ tổng **8**.
- Với `k = 1`: mua 1 bán 9: `8 − 2 = 6`.

Lưu ý: với phí, phím tắt `k ≥ n/2` vẫn đúng không? Hãy suy nghĩ (gợi ý ở mục 17).

In [37]:
def exercise_k_with_fee(k, prices, fee):
    # TODO: sửa đúng một dòng của Cách 4 để trừ phí khi bán
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# assert exercise_k_with_fee(2, [1, 3, 2, 8, 4, 9], 2) == 8
# assert exercise_k_with_fee(1, [1, 3, 2, 8, 4, 9], 2) == 6
# assert exercise_k_with_fee(3, [1, 2], 5) == 0

### Bài 6 — Mini challenge: in ra **các giao dịch** cụ thể

LeetCode chỉ hỏi con số, nhưng nhà đầu tư muốn biết **mua ngày nào, bán ngày nào**. Viết `exercise_transactions(k, prices)` trả về `(profit, trades)` với `trades` là list các cặp `(ngày_mua, ngày_bán)`, sao cho:

- `len(trades) <= k`, các giao dịch không chồng nhau (`bán_trước <= mua_sau`), mỗi cặp có `mua < bán`;
- tổng `prices[bán] − prices[mua]` đúng bằng `profit` tối ưu.

Gợi ý: dùng bảng `dp[t][i]` của Cách 3b và lưu thêm **`j` nào** cho ra `best`, rồi **lần ngược** (backtrack) từ `dp[k][n-1]`.

In [38]:
def exercise_transactions(k, prices):
    # TODO 1: tính bảng dp[t][i] như Cách 3b, kèm best_j[t][i] = ngày mua tốt nhất khi bán ở i
    # TODO 2: lần ngược từ (t=k, i=n-1): nếu dp[t][i] == dp[t][i-1] thì lùi i; ngược lại ghi giao dịch
    raise NotImplementedError

# Tự làm xong rồi mới mở test (dùng hàm kiểm tra trong phần đáp án):
# check_trades(2, [3, 2, 6, 5, 0, 3], *exercise_transactions(2, [3, 2, 6, 5, 0, 3]))

## 17. Gợi ý theo mức

**Bài 1**
- Mức 1: Ngày đầu tiên, `buy[t] = sell[t-1] − giá` vì `buy` cũ là `−∞`.
- Mức 2: Ngày 2 (giá 2), `buy[2] = sell[1] − 2`, mà `sell[1]` hôm trước đã là 2.
- Mức 3: Kết quả cuối `k = 2` là `2 + 6`; `k = 1` là `8 − 1`.

**Bài 2**
- Mức 1: Câu 1 giá không đổi; câu 2 đếm đoạn tăng.
- Mức 2: Câu 2 có `n = 6`, `n // 2 = 3`, mà `k = 5`.
- Mức 3: Câu 5: so "mua 0 bán 10" với "mua 1 bán 9".

**Bài 3**
- Mức 1: Xem lại "ví tiền" ở mục 9.
- Mức 2: `[float("-inf")] * (k + 1)` và `[0] * (k + 1)`.
- Mức 3: `buy[t] = max(buy[t], sell[t-1] - p)`; `sell[t] = max(sell[t], buy[t] + p)`; `return sell[k]`.

**Bài 4**
- Mức 1: Mua cần `t > 0` và gọi `f(i+1, t-1, True)`; bán gọi `f(i+1, t, False)`.
- Mức 2: Base case `i == n` ⇒ `0`. Có thể thêm `if t == 0 and not holding: return 0` để cắt nhánh.
- Mức 3: Khi `holding` thì luôn được bán, bất kể `t`.

**Bài 5**
- Mức 1: Phí trừ đúng **một lần** mỗi giao dịch — gắn vào lúc bán.
- Mức 2: `sell[t] = max(sell[t], buy[t] + p - fee)`.
- Mức 3: Phím tắt vẫn đúng ý tưởng (số giao dịch có lãi ≤ số đoạn tăng ≤ n//2), nhưng "cộng mọi bước tăng" thì **sai** khi có phí (vì trả phí nhiều lần). Đơn giản nhất: bỏ phím tắt, hoặc dùng `min(k, n // 2)` làm số lượt cho vòng `buy/sell`.

**Bài 6**
- Mức 1: Khi cập nhật `best = max(best, dp[t-1][i] - prices[i])`, nhớ luôn `i` nào cho ra `best`.
- Mức 2: Lúc lần ngược, nếu `dp[t][i] != dp[t][i-1]` thì ngày `i` là ngày bán của giao dịch `t`, ngày mua là `best_j[t][i]`; sau đó nhảy tới `(t−1, j)`.
- Mức 3: Dừng khi `t == 0` hoặc `i == 0`; nhớ đảo ngược danh sách vì ta thu thập từ cuối về đầu.

## 18. Đáp án tham khảo

### Đáp án Bài 1

| Ngày | Giá | `buy[1]` | `sell[1]` | `buy[2]` | `sell[2]` |
|---:|---:|---:|---:|---:|---:|
| 0 | 1 | −1 | 0 | −1 | 0 |
| 1 | 3 | −1 | 2 | −1 | 2 |
| 2 | 2 | −1 | 2 | 0 | 2 |
| 3 | 8 | −1 | 7 | 0 | **8** |

`k = 2` ⇒ **8** (mua 1 bán 3, mua 2 bán 8). `k = 1` ⇒ **7** (mua 1 bán 8).

### Đáp án Bài 2

1. `0` — giá không đổi, không lời.
2. `3` — đi qua **phím tắt** (`k = 5 ≥ 6 // 2 = 3`), cộng ba bước tăng.
3. `7` — gộp: mua 1 bán 8.
4. `8` — tách: 1→5 và 4→8.
5. `10` — mua 0 bán 10 (lời 10) tốt hơn mua 1 bán 9 (lời 8).

Cell dưới kiểm chứng Bài 1 và Bài 2 bằng code đã viết.

In [39]:
trace_buy_sell(2, [1, 3, 2, 8])
assert max_profit_buy_sell(1, [1, 3, 2, 8]) == 7

answers_2 = [((1, [3, 3, 3]), 0), ((5, [1, 2, 1, 2, 1, 2]), 3), ((1, [1, 5, 4, 8]), 7),
             ((2, [1, 5, 4, 8]), 8), ((1, [5, 1, 9, 0, 10]), 10)]
for (k, p), exp in answers_2:
    assert Solution().maxProfit(k, p) == exp, (k, p)
print("Bài 1, Bài 2: PASS")

ngày | giá | buy[1] sell[1] | buy[2] sell[2]
   0 |   1 |     -1       0 |     -1       0
   1 |   3 |     -1       2 |     -1       2
   2 |   2 |     -1       2 |      0       2
   3 |   8 |     -1       7 |      0       8
Đáp án sell[k] = 8
Bài 1, Bài 2: PASS


### Đáp án Bài 3 và Bài 4

In [40]:
def answer_buy_sell(k, prices):
    buy = [float("-inf")] * (k + 1)
    sell = [0] * (k + 1)
    for p in prices:
        for t in range(1, k + 1):
            buy[t] = max(buy[t], sell[t - 1] - p)
            sell[t] = max(sell[t], buy[t] + p)
    return sell[k]


def answer_memo_count_on_buy(k, prices):
    n = len(prices)
    sys.setrecursionlimit(max(sys.getrecursionlimit(), 2 * n + 100))

    @lru_cache(maxsize=None)
    def f(i, t, holding):
        if i == n:
            return 0
        if t == 0 and not holding:          # hết lượt mua và không giữ gì -> hết cơ hội
            return 0
        skip = f(i + 1, t, holding)
        if holding:
            act = prices[i] + f(i + 1, t, False)        # bán: không trừ lượt
        else:
            act = -prices[i] + f(i + 1, t - 1, True)    # mua: trừ lượt (t > 0 chắc chắn)
        return max(skip, act)

    return f(0, k, False)


for fn in (answer_buy_sell, answer_memo_count_on_buy):
    assert fn(2, [2, 4, 1]) == 2
    assert fn(2, [3, 2, 6, 5, 0, 3]) == 7
    assert fn(1, [1, 5, 4, 8]) == 7
print("Bài 3, Bài 4: PASS")

Bài 3, Bài 4: PASS


### Đáp án Bài 5

Chỉ sửa dòng bán thành `buy[t] + p - fee`. Không dùng phím tắt "cộng mọi bước tăng" vì có phí thì phép cộng đó sai; thay vào đó giới hạn số lượt bằng `min(k, n // 2)` (vẫn đúng vì số giao dịch có lãi không vượt quá số đoạn tăng). Kiểm chứng bằng một brute force đệ quy có phí.

In [41]:
def answer_k_with_fee(k, prices, fee):
    k = min(k, len(prices) // 2)
    buy = [float("-inf")] * (k + 1)
    sell = [0] * (k + 1)
    for p in prices:
        for t in range(1, k + 1):
            buy[t] = max(buy[t], sell[t - 1] - p)
            sell[t] = max(sell[t], buy[t] + p - fee)     # dòng duy nhất thay đổi
    return sell[k]


def brute_k_with_fee(k, prices, fee):
    n = len(prices)
    def f(i, t, holding):
        if i == n or t == 0:
            return 0
        skip = f(i + 1, t, holding)
        act = prices[i] - fee + f(i + 1, t - 1, False) if holding else -prices[i] + f(i + 1, t, True)
        return max(skip, act)
    return f(0, k, False)


assert answer_k_with_fee(2, [1, 3, 2, 8, 4, 9], 2) == 8
assert answer_k_with_fee(1, [1, 3, 2, 8, 4, 9], 2) == 6
assert answer_k_with_fee(3, [1, 2], 5) == 0
random.seed(55)
for _ in range(500):
    n = random.randint(1, 9)
    pr = [random.randint(0, 10) for _ in range(n)]
    kk, fee = random.randint(1, 5), random.randint(0, 3)
    assert answer_k_with_fee(kk, pr, fee) == brute_k_with_fee(kk, pr, fee), (kk, pr, fee)
print("Bài 5: PASS (kể cả 500 test ngẫu nhiên so với brute force)")

Bài 5: PASS (kể cả 500 test ngẫu nhiên so với brute force)


### Đáp án Bài 6

`best_j[t][i]` lưu ngày mua `j` đang cho giá trị `best` tốt nhất khi xét bán ở ngày `i`. Khi lần ngược, mỗi lần `dp[t][i]` khác `dp[t][i-1]` nghĩa là "ngày `i` có bán" — ta ghi giao dịch `(j, i)` rồi nhảy về `(t−1, j)` vì phần trước đó dùng `dp[t-1][j]`.

Hàm `check_trades` kiểm tra các điều kiện hợp lệ mà đề yêu cầu (thay vì so khớp một list cụ thể, vì có thể có nhiều phương án tối ưu).

In [42]:
def answer_transactions(k, prices):
    n = len(prices)
    if n == 0 or k == 0:
        return 0, []
    dp = [[0] * n for _ in range(k + 1)]
    best_j = [[0] * n for _ in range(k + 1)]
    for t in range(1, k + 1):
        best, bj = dp[t - 1][0] - prices[0], 0
        for i in range(1, n):
            best_j[t][i] = bj                      # ngày mua tốt nhất nếu bán ở i
            dp[t][i] = max(dp[t][i - 1], prices[i] + best)
            if dp[t - 1][i] - prices[i] > best:
                best, bj = dp[t - 1][i] - prices[i], i
    trades = []
    t, i = k, n - 1
    while t > 0 and i > 0:
        if dp[t][i] == dp[t][i - 1]:
            i -= 1                                 # ngày i không bán
        else:
            j = best_j[t][i]
            trades.append((j, i))                  # mua ở j, bán ở i
            t, i = t - 1, j
    trades.reverse()
    return dp[k][n - 1], trades


def check_trades(k, prices, profit, trades):
    assert profit == max_profit_buy_sell(k, prices), "profit chưa tối ưu"
    assert len(trades) <= k, "quá k giao dịch"
    for (b, s) in trades:
        assert b < s, "phải mua trước bán sau"
    for (b1, s1), (b2, s2) in zip(trades, trades[1:]):
        assert s1 <= b2, "giao dịch chồng nhau"
    assert sum(prices[s] - prices[b] for b, s in trades) == profit, "tổng lời không khớp"


print(answer_transactions(2, [3, 2, 6, 5, 0, 3]))       # (7, [(1, 2), (4, 5)])
print(answer_transactions(1, [1, 5, 4, 8]))             # (7, [(0, 3)])
print(answer_transactions(3, [1, 2, 4, 2, 5, 7, 2, 4, 9, 0]))
random.seed(66)
for _ in range(500):
    n = random.randint(1, 10)
    pr = [random.randint(0, 10) for _ in range(n)]
    kk = random.randint(1, 5)
    check_trades(kk, pr, *answer_transactions(kk, pr))
print("Bài 6: PASS (500 test ngẫu nhiên)")

(7, [(1, 2), (4, 5)])
(7, [(0, 3)])
(15, [(0, 2), (3, 5), (6, 8)])
Bài 6: PASS (500 test ngẫu nhiên)


## 19. Template ngắn nên nhớ

```python
class Solution:
    def maxProfit(self, k: int, prices: List[int]) -> int:
        n = len(prices)
        if k >= n // 2:                                   # không giới hạn thực sự -> #122
            return sum(max(0, prices[i] - prices[i - 1]) for i in range(1, n))
        buy = [float("-inf")] * (k + 1)                   # đã mua lần t, đang giữ
        sell = [0] * (k + 1)                              # đã bán lần t, cầm tiền
        for p in prices:
            for t in range(1, k + 1):
                buy[t] = max(buy[t], sell[t - 1] - p)     # giữ, hoặc mua hôm nay
                sell[t] = max(sell[t], buy[t] + p)        # giữ, hoặc bán hôm nay
        return sell[k]
```

Câu thần chú: **"Mua lần `t` lấy tiền từ bán lần `t−1`; bán lần `t` lấy cổ phiếu từ mua lần `t`. `buy` bắt đầu từ âm vô cực, `sell` bắt đầu từ 0. `k` lớn quá thì cộng mọi bước tăng."**

## 20. Checklist trước khi submit

- `buy` khởi tạo `float("-inf")` (không phải 0), `sell` khởi tạo `0` (không phải `-inf`)?
- Dòng bán dùng `buy[t]` (cùng chỉ số), dòng mua dùng `sell[t-1]` (chỉ số trước)?
- Vòng `t` chạy `range(1, k + 1)` — không quên `k`, không đụng `t = 0`?
- Có phím tắt `k >= n // 2` để tránh vòng `k` thừa (và tránh tràn bộ nhớ nếu `k` khổng lồ)?
- Trả về `sell[k]`, không âm; đã thử `n = 1`, mảng giảm dần, giá không đổi, `[1,5,4,8]` với `k = 1` và `k = 2`, và hai example của đề?
- Nếu dùng đệ quy + memo: đã `sys.setrecursionlimit` cho `n = 1000` chưa?